In [0]:
%run ./utils

# Imports

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

# Load

In [0]:

dbutils.widgets.text("bronze_table_name", "tb_custos")
bronze_table_name = dbutils.widgets.get("bronze_table_name")

In [0]:
df_custos = (
    spark.table(f"{bronze_schema}.{bronze_table_name}")
        .distinct()
        .orderBy("id_custo")
)

In [0]:
df_custos.count()

# Deduplicação

In [0]:
# Definição da janela agrupada pela chave primária e ordenada pelo timestamp decrescente
window_spec = Window.partitionBy("id_custo").orderBy(F.col("ingestion_datetime").desc())

In [0]:
df_dedup = (
    df_custos
    .withColumn("rn", F.row_number().over(window_spec))
    .filter(F.col("rn") == 1)
    .drop("rn", "ingestion_datetime")
)

# Tratamento

In [0]:
# cast dos ids
df_cast = cast_id_to_string(df_dedup)

In [0]:
# Elimina apenas moeda (R$ e reais)
df_limpo = df_cast.withColumn("custo",F.trim(F.regexp_replace(F.lower(F.col("custo")), r"r\$|reais|real", "")))

In [0]:
# Troca "","" para marcador universal
df_num = df_limpo.withColumn("custo",F.regexp_replace(F.col("custo"),",","."))

In [0]:
# cast dos custos para decimal
df_decimal = df_num.withColumn("custo",F.col("custo").cast("decimal(18,2)"))

# Verificação de Qualidade

In [0]:
# Unicidade de chave composta
dq_check_unique("silver.tb_custos", "chave_composta_unica", df_decimal, ["id_chamado", "id_custo"])

In [0]:
# Confere persistencia de custos negativos
dq_check(
    "silver.tb_custos",
    "negativos_preservados",
    df_decimal.join(
        df_custos.select("id_custo", "id_chamado", F.col("custo").contains("-").alias("neg_bronze")),
        ["id_custo", "id_chamado"],
        "left"
    ),
    (F.col("custo") < 0) == F.col("neg_bronze")
)

# Salvamento

In [0]:
dbutils.widgets.text("silver_table_name", "tb_custos")
silver_table_name = dbutils.widgets.get("silver_table_name")

In [0]:
save_df(df_decimal, silver_table_name)